In [1]:
import polars as pl

df = pl.read_csv("data/olympics_athletes_dataset.csv")

In [2]:
df.head()


athlete_id,athlete_name,gender,age,date_of_birth,nationality,country_name,sport,event,games_type,year,host_city,team_or_individual,medal,result_value,result_unit,total_olympics_attended,total_medals_won,gold_medals,silver_medals,bronze_medals,country_total_gold,country_total_medals,country_first_participation,country_best_rank,is_record_holder,coach_name,height_cm,weight_kg,notes
str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,f64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,f64,f64,str
"""ATH-00001""","""Svetlana Jung""","""Female""",19,"""2005-12-04""","""AUT""","""Austria""","""Rowing""","""Four W""","""Summer""",1896,"""Athens""","""Team""","""No Medal""",494.699,"""seconds""",1,8,1,6,1,59,196,1896,18,"""No""","""Wei Ping""",175.9,73.7,"""-"""
"""ATH-00002""","""Mary Yamamoto""","""Female""",37,"""1987-07-11""","""MEX""","""Mexico""","""Ski Jumping""","""Normal Hill Team""","""Winter""",1960,"""Squaw Valley""","""Individual""","""Silver""",132.463,"""metres""",4,6,0,1,5,14,72,1924,35,"""No""","""Yury Zakharevich""",165.4,68.3,"""Olympic Debut"""
"""ATH-00003""","""Oksana Volkov""","""Female""",37,"""1987-02-02""","""BUL""","""Bulgaria""","""Figure Skating""","""Women's Singles""","""Winter""",1932,"""Lake Placid""","""Individual""","""No Medal""",215.67,"""points""",4,1,0,0,1,54,224,1896,15,"""No""","""Alberto Salazar""",164.2,67.2,"""-"""
"""ATH-00004""","""Rui Suzuki""","""Male""",32,"""1992-12-08""","""HKG""","""Hong Kong""","""Triathlon""","""Men's Triathlon""","""Summer""",2012,"""London""","""Individual""","""No Medal""",4470.383,"""seconds""",3,4,4,0,0,3,9,1952,60,"""Olympic Record""","""Marcus O'Sullivan""",190.0,76.0,"""First from country"""
"""ATH-00005""","""Natalya Grigoryan""","""Female""",27,"""1997-11-15""","""SWE""","""Sweden""","""Triathlon""","""Men's Triathlon""","""Summer""",1900,"""Paris""","""Individual""","""No Medal""",7390.643,"""seconds""",5,3,2,1,0,200,648,1896,4,"""No""","""John Smith""",175.8,60.9,"""Season Best"""


In [3]:
df.columns

['athlete_id',
 'athlete_name',
 'gender',
 'age',
 'date_of_birth',
 'nationality',
 'country_name',
 'sport',
 'event',
 'games_type',
 'year',
 'host_city',
 'team_or_individual',
 'medal',
 'result_value',
 'result_unit',
 'total_olympics_attended',
 'total_medals_won',
 'gold_medals',
 'silver_medals',
 'bronze_medals',
 'country_total_gold',
 'country_total_medals',
 'country_first_participation',
 'country_best_rank',
 'is_record_holder',
 'coach_name',
 'height_cm',
 'weight_kg',
 'notes']

In [4]:
pip install vaderSentiment

Note: you may need to restart the kernel to use updated packages.


In [5]:
import polars as pl

df = pl.read_csv("data/olympics_athletes_dataset.csv")

df = df.with_columns([
    pl.col("notes").cast(pl.Utf8),
    # won medal if medal is not null and not empty and not "None"/"NA" variants
    (
        pl.col("medal")
        .cast(pl.Utf8)
        .fill_null("")
        .str.strip_chars()
        .str.to_lowercase()
        .is_in(["gold", "silver", "bronze"])
    ).alias("won_medal")
])

df.select(["notes", "medal", "won_medal"]).head(10)

notes,medal,won_medal
str,str,bool
"""-""","""No Medal""",false
"""Olympic Debut""","""Silver""",true
"""-""","""No Medal""",false
"""First from country""","""No Medal""",false
"""Season Best""","""No Medal""",false
"""Youngest competitor""","""No Medal""",false
"""-""","""No Medal""",false
"""Disqualified (false start)""","""No Medal""",false
"""-""","""No Medal""",false


In [6]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Keep only rows with notes
df_notes = df.filter(pl.col("notes").is_not_null() & (pl.col("notes").str.len_chars() > 0))

# VADER scores
df_notes = df_notes.with_columns([
    pl.col("notes").map_elements(lambda s: analyzer.polarity_scores(s)["compound"], return_dtype=pl.Float64).alias("vader_compound"),
    pl.col("notes").map_elements(lambda s: analyzer.polarity_scores(s)["pos"], return_dtype=pl.Float64).alias("vader_pos"),
    pl.col("notes").map_elements(lambda s: analyzer.polarity_scores(s)["neu"], return_dtype=pl.Float64).alias("vader_neu"),
    pl.col("notes").map_elements(lambda s: analyzer.polarity_scores(s)["neg"], return_dtype=pl.Float64).alias("vader_neg"),
])

df_notes.select(["notes", "vader_compound", "won_medal"]).head(10)

notes,vader_compound,won_medal
str,f64,bool
"""-""",0.0,false
"""Olympic Debut""",0.0,true
"""-""",0.0,false
"""First from country""",0.0,false
"""Season Best""",0.6369,false
"""Youngest competitor""",0.0,false
"""-""",0.0,false
"""Disqualified (false start)""",-0.4215,false
"""-""",0.0,false


In [8]:
df_notes.group_by("won_medal").agg([
    pl.count().alias("n"),
    pl.mean("vader_compound").alias("avg_compound"),
    pl.median("vader_compound").alias("median_compound"),
    pl.mean("vader_pos").alias("avg_pos"),
    pl.mean("vader_neg").alias("avg_neg"),
]).sort("won_medal")

C:\Users\im4284hw\AppData\Local\Temp\ipykernel_2968\4148996085.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("n"),


won_medal,n,avg_compound,median_compound,avg_pos,avg_neg
bool,u32,f64,f64,f64,f64
false,6491,-0.037865,0.0,0.070331,0.125811
true,2009,-0.039042,0.0,0.067166,0.124497


In [9]:
import re
import pandas as pd

# Convert needed columns to pandas
pdf = df_notes.select(["notes", "won_medal", "vader_compound"]).to_pandas()

# Basic tokenizer
STOPWORDS = {
    "the","a","an","and","or","to","of","in","on","for","with","at","by",
    "from","is","was","were","as","it","this","that","their","his","her",
    "first","olympic","olympics","debut"  # optionally remove very common domain words
}

def tokenize(text: str):
    text = text.lower()
    # keep simple words; you can expand to keep bigrams later
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text)
    return [w for w in words if len(w) >= 3 and w not in STOPWORDS]

rows = []
for notes, won, comp in zip(pdf["notes"], pdf["won_medal"], pdf["vader_compound"]):
    if not isinstance(notes, str) or not notes.strip():
        continue
    uniq_words = set(tokenize(notes))
    for w in uniq_words:  # unique per row so counts reflect "docs containing word"
        rows.append((w, int(won), float(comp)))

wd = pd.DataFrame(rows, columns=["word", "won_medal", "compound"])

overall_rate = pdf["won_medal"].mean()

word_stats = (
    wd.groupby("word")
      .agg(
          count=("word", "size"),
          medal_rate=("won_medal", "mean"),
          avg_compound=("compound", "mean")
      )
      .assign(lift=lambda d: d["medal_rate"] - overall_rate)
      .sort_values(["lift", "count"], ascending=[False, False])
)

# Filter to avoid rare words
word_stats_filtered = word_stats.query("count >= 50").copy()

# Top words associated with higher medal rate
top_medal_words = word_stats_filtered.sort_values(["lift","count"], ascending=[False, False]).head(25)
top_medal_words

,count,medal_rate,avg_compound,lift
word,,,,
national,376,0.276596,0.000000,0.040243
finish,350,0.265714,0.000000,0.029361
photo,350,0.265714,0.000000,0.029361
silver,350,0.265714,0.000000,0.029361
equipment,380,0.255263,0.000000,0.018910
issue,380,0.255263,0.000000,0.018910
record,730,0.250685,-0.231167,0.014332
appearance,336,0.250000,0.000000,0.013647
final,336,0.250000,0.000000,0.013647


In [10]:
positive_medal_words = (
    word_stats_filtered
      .query("avg_compound >= 0.20")           # positive-ish
      .sort_values(["medal_rate","avg_compound","count"], ascending=[False, False, False])
      .head(25)
)
positive_medal_words

,count,medal_rate,avg_compound,lift
word,,,,
personal,391,0.235294,0.6369,-0.001059
best,732,0.228142,0.6369,-0.008211
season,341,0.219941,0.6369,-0.016412


In [11]:
import polars as pl

df = pl.read_csv("data/olympics_athletes_dataset.csv")

df = df.with_columns([
    pl.col("notes").cast(pl.Utf8),
    pl.col("notes")
      .fill_null("")
      .str.strip_chars()
      .str.to_lowercase()
      .alias("phrase"),

    (
        pl.col("medal")
        .cast(pl.Utf8)
        .fill_null("")
        .str.strip_chars()
        .str.to_lowercase()
        .is_in(["gold", "silver", "bronze"])
    ).alias("won_medal")
])

df_phrases = df.filter(pl.col("phrase") != "")

In [12]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

df_phrases = df_phrases.with_columns([
    pl.col("phrase").map_elements(
        lambda s: analyzer.polarity_scores(s)["compound"],
        return_dtype=pl.Float64
    ).alias("vader_compound")
])

In [14]:
phrase_stats = (
    df_phrases
    .group_by("phrase")
    .agg([
        pl.len().alias("count"),
        pl.col("won_medal").cast(pl.Float64).mean().alias("medal_rate"),
        pl.col("vader_compound").mean().alias("avg_sentiment"),
    ])
    .sort("count", descending=True)
)

phrase_stats.head(20)

phrase,count,medal_rate,avg_sentiment
str,u32,f64,f64
"""-""",2799,0.231511,0.0
"""personal best""",391,0.235294,0.6369
"""equipment issue""",380,0.255263,0.0
"""olympic debut""",377,0.228117,0.0
"""national record""",376,0.276596,0.0
…,…,…,…
"""technical violation""",343,0.239067,-0.4939
"""season best""",341,0.219941,0.6369
"""first from country""",340,0.229412,0.0


In [15]:
overall_medal_rate = (
    df_phrases
    .select(pl.col("won_medal").cast(pl.Float64).mean())
    .item()
)

phrase_stats = phrase_stats.with_columns(
    (pl.col("medal_rate") - overall_medal_rate).alias("lift_vs_baseline")
)

phrase_stats.sort("lift_vs_baseline", descending=True).head(20)

phrase,count,medal_rate,avg_sentiment,lift_vs_baseline
str,u32,f64,f64,f64
"""national record""",376,0.276596,0.0,0.040243
"""photo-finish silver""",350,0.265714,0.0,0.029361
"""equipment issue""",380,0.255263,0.0,0.01891
"""final appearance""",336,0.25,0.0,0.013647
"""weather-affected""",360,0.247222,0.0,0.010869
…,…,…,…,…
"""comeback after injury""",361,0.227147,-0.4215,-0.009206
"""record broken next day""",354,0.223164,-0.4767,-0.013189
"""season best""",341,0.219941,0.6369,-0.016412


Use this one ....

In [18]:
top5 = (
    phrase_stats
    .filter(pl.col("count") >= 50)   # adjust threshold if needed
    .sort("lift_vs_baseline", descending=True)
    .select(["phrase", "count", "medal_rate", "lift_vs_baseline"])
    .head(5)
)

top5

phrase,count,medal_rate,lift_vs_baseline
str,u32,f64,f64
"""national record""",376,0.276596,0.040243
"""photo-finish silver""",350,0.265714,0.029361
"""equipment issue""",380,0.255263,0.01891
"""final appearance""",336,0.25,0.013647
"""weather-affected""",360,0.247222,0.010869


not this 

In [17]:
top5_positive = (
    phrase_stats
    .filter((pl.col("count") >= 50) & (pl.col("avg_sentiment") > 0.2))
    .sort("medal_rate", descending=True)
    .select(["phrase", "count", "medal_rate", "avg_sentiment", "lift_vs_baseline"])
    .head(5)
)

top5_positive

phrase,count,medal_rate,avg_sentiment,lift_vs_baseline
str,u32,f64,f64,f64
"""personal best""",391,0.235294,0.6369,-0.001059
"""season best""",341,0.219941,0.6369,-0.016412
